# OpenVINO Physical AI APIs enabling deployment of optimized model

### Minimal code to run a Pi0.5 model on an SO101 robot

<img src="media/2. studio+OV.png" width="800">

## 1) Train a model using the Physical AI Studio or LeRobot
* Use Pi0.5 policy supported in Physical AI Studio or LeRobot to train a model.<br>
* The output of this stage should be a trained Pi0.5 model<br>
<font size="2">[Open Notebook](002_Using_Physical_AI_Studio.ipynb)

## 2) Install OpenVINO Physical AI on your target platform

Clone the OpenVINO Physical AI repo and install a pinned revision.

> **Why pin the revision?** The OpenVINO Physical AI APIs are evolving quickly. This notebook depends on a tested `physicalai` runtime revision, so the install cell below checks out a known-good commit before installing the package in editable mode. Update `PHYSICALAI_COMMIT` only after re-validating the rest of this notebook.

In [ ]:
from pathlib import Path
import subprocess
import sys

requirements_file = Path("requirements.txt")
if not requirements_file.exists():
    requirements_file = Path("notebooks/requirements.txt")

# Tested with this notebook. Pinning avoids breakage from fast-moving API changes.
# The install flow checks out the commit directly so it does not depend on the
# docs/tutorials branch name after these notebooks are merged into the default branch.
PHYSICALAI_REPO = "https://github.com/openvinotoolkit/physicalai.git"
PHYSICALAI_REF = "docs/tutorials"
PHYSICALAI_COMMIT = "6f764b8b4534ad32bf2fdacea066e19cf0921cb0"
physicalai_dir = Path("physicalai")

if not physicalai_dir.exists():
    subprocess.check_call(["git", "clone", PHYSICALAI_REPO, str(physicalai_dir)])

has_pinned_commit = subprocess.run(
    ["git", "-C", str(physicalai_dir), "cat-file", "-e", f"{PHYSICALAI_COMMIT}^{{commit}}"],
    check=False,
).returncode == 0
if not has_pinned_commit:
    subprocess.check_call(["git", "-C", str(physicalai_dir), "fetch", "origin", PHYSICALAI_REF])

subprocess.check_call(["git", "-C", str(physicalai_dir), "checkout", PHYSICALAI_COMMIT])
print("Using physicalai commit:")
subprocess.check_call(["git", "-C", str(physicalai_dir), "rev-parse", "--short", "HEAD"])

%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu -r {requirements_file}
%pip install -q -e physicalai


Import libraries, and set up a working directory

In [ ]:
from pathlib import Path
import json
import os
import time

import numpy as np
import openvino as ov
from huggingface_hub import hf_hub_download, snapshot_download
from physicalai.inference import InferenceModel

WORKSPACE = Path.cwd().resolve()
RUNTIME_ROOT = WORKSPACE / "physicalai"
ROOT = RUNTIME_ROOT
os.chdir(ROOT)

Install extras, mainly for hardware acceleration (e.g. for SO101 or specific camera types)

In [ ]:
INSTALL_HARDWARE_DEPS = "True"
USE_SHARED_CAMERA = "True"

if INSTALL_HARDWARE_DEPS:
    import subprocess
    import sys

    hardware_extras = "transport,so101" if USE_SHARED_CAMERA else "so101"
    runtime_with_hardware_extras = str(WORKSPACE / "physicalai") + f"[{hardware_extras}]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", runtime_with_hardware_extras])
    print(f"[DONE] Installed PhysicalAI hardware extras: {hardware_extras}")
else:
    print("[SKIP] Hardware extras are not installed. Set INSTALL_HARDWARE_DEPS = True if this environment needs them.")


## 3) Discover and connect to Robot and Cameras

> **Before continuing on Linux:** make sure the user running Jupyter has permission to access USB cameras and the SO101 serial device. If the robot appears as `/dev/ttyACM0`, the user usually needs to be in the `dialout` group. Camera access may require membership in the `video` group.

In [ ]:
# Optional Linux permission check for cameras and the SO101 serial device.
# If the current user is missing the required groups, run the printed commands
# in a terminal, then log out/in or restart the machine before continuing.
import getpass
import os
from pathlib import Path

try:
    import grp
except ImportError:
    grp = None

user = getpass.getuser()
print("Current user:", user)

if grp is None:
    print("Group membership checks are only available on Unix-like systems.")
else:
    groups = {g.gr_name for g in grp.getgrall() if user in g.gr_mem}
    try:
        groups.add(grp.getgrgid(os.getgid()).gr_name)
    except KeyError:
        pass
    print("Groups:", " ".join(sorted(groups)))

    for group in ["video", "dialout"]:
        if group not in groups:
            print(f"[ACTION REQUIRED] Add {user!r} to the {group!r} group:")
            print(f"  sudo usermod -aG {group} {user}")

serial_port = Path("/dev/ttyACM0")
if serial_port.exists():
    print("/dev/ttyACM0 mode/owner:", oct(serial_port.stat().st_mode), serial_port.stat().st_uid, serial_port.stat().st_gid)
    if not os.access(serial_port, os.R_OK | os.W_OK):
        print("[ACTION REQUIRED] Current process cannot read/write /dev/ttyACM0.")
        print("After adding the user to dialout, restart the login session/Jupyter server.")
else:
    print("/dev/ttyACM0 was not found. Update SO101_PORT later if your robot uses a different serial device.")


<div style="font-size: 18px; font-weight: 700;">
Discover cameras.
</div>


In [ ]:
# Discover available cameras — use the device IDs from this output in the config below
from physicalai.capture import discover_all

for driver, devices in discover_all().items():
    if not devices:
        continue
    print(f"\n[{driver}]")
    for dev in devices:
        print(f"  {dev.device_id}  —  {dev.name}")


<div style="font-size: 16px; font-weight: 700;">
ACTION REQUIRED: Configure your cameras before running the next code cell.
</div>

<div style="font-size: 18px;">
Use the device IDs printed by the previous cell. Keep the camera names exactly the same as the names used during dataset collection and model export.
</div>

For the released Pi0.5 pick-and-place model used in this tutorial, the model expects **two** cameras. Do not add a third camera such as `table-cam` unless your exported model was trained with that camera.

Expected names for the example model:

- `top-cam`
- `gripper-cam`

<img src="media/detect-cameras.png" alt="Camera discovery output" width="800" height="200">


In [ ]:
# Cameras - use the same names and setup as used for dataset collection and model export.
# ============================================================
# ACTION REQUIRED - Edit these before running.
# ============================================================
# Use the LEFT column from the camera discovery output as device_id.
# For UVC cameras, prefer stable /dev/v4l/by-id/... paths instead of numeric IDs.
#
# The example Pi0.5 model expects exactly two cameras:
#   - "top-cam"
#   - "gripper-cam"
# Remove or rename cameras only if your exported model was trained with different names.
CAMERAS = [
    ("top-cam", "uvc", "/dev/v4l/by-id/<your-top-camera-id>"),
    ("gripper-cam", "uvc", "/dev/v4l/by-id/<your-gripper-camera-id>"),
]

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
CAMERA_FPS = 30

# Runtime
FPS = 30
DURATION_S = 60.0


Define robot and get the robot ID from Physical AI Studio.

<img src="media/robot-ID.png" alt="Robot ID in Physical AI Studio" width="800" height="267">

<div style="font-size: 16px; font-weight: 700;">
ACTION REQUIRED: Configure the SO101 serial port and calibration file before connecting the robot.
</div>

The calibration file path depends on how you collected data:

- Physical AI Studio: `~/.cache/physicalai/robots/<robot-id>/calibrations/<cal-id>.json`
- LeRobot: `~/.cache/calibration/so101_follower.json`

Make sure the Jupyter user can read the calibration JSON. If you see `PermissionError`, fix file ownership/permissions or copy the calibration file to a readable path.


In [ ]:
# ============================================================
# ACTION REQUIRED - Edit these before running.
# ============================================================
# Robot serial port. Common Linux values are /dev/ttyACM0 or /dev/ttyUSB0.
SO101_PORT = "/dev/ttyACM0"


In [ ]:
# ============================================================
# ACTION REQUIRED - Edit this calibration path before running the connect cell.
# ============================================================
# Physical AI Studio example:
# SO101_CALIBRATION = "~/.cache/physicalai/robots/<robot-id>/calibrations/<cal-id>.json"
#
# LeRobot example:
# SO101_CALIBRATION = "~/.cache/calibration/so101_follower.json"
SO101_CALIBRATION = "~/.cache/physicalai/robots/<robot-id>/calibrations/<cal-id>.json"

SO101_CALIBRATION = str(Path(SO101_CALIBRATION).expanduser())
if "<" in SO101_CALIBRATION or ">" in SO101_CALIBRATION:
    raise ValueError("Update SO101_CALIBRATION to your actual calibration JSON path before continuing.")
if not Path(SO101_CALIBRATION).is_file():
    raise FileNotFoundError(f"Calibration file not found: {SO101_CALIBRATION}")
print("Using SO101 calibration:", SO101_CALIBRATION)


Connect cameras and robot together. This sets up the synchronized robot/camera runtime environment.

<div style="font-size: 16px; font-weight: 700;">
ACTION REQUIRED: Confirm cameras, robot connection, and permissions before running the next code cell.
</div>

Before running this cell:

- Make sure Physical AI Studio and other camera applications are closed.
- Confirm the current user can access the SO101 serial port, for example `/dev/ttyACM0`.
- Confirm `CAMERAS` contains exactly the camera names expected by your exported model. The example model expects `top-cam` and `gripper-cam` only.
- If a previous connection attempt failed, disconnect cameras or restart the kernel before trying again.


In [ ]:
from physicalai.capture import SharedCamera
from physicalai.robot import SO101

# Cameras
cameras = {}
try:
    for name, driver, device_id in CAMERAS:
        kwargs = {"serial_number": device_id} if driver == "realsense" else {"device": device_id}
        cam = SharedCamera(driver, **kwargs, width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=CAMERA_FPS)
        cam.connect()
        cameras[name] = cam
        print(f"Camera '{name}' connected: {cam.actual_width}x{cam.actual_height} @ {cam.actual_fps}fps")

    # Robot
    robot = SO101(port=SO101_PORT, calibration=SO101_CALIBRATION, role="follower")
    robot.connect()
    print(f"Robot connected on {SO101_PORT}")
except Exception:
    for cam in cameras.values():
        try:
            cam.disconnect()
        except Exception:
            pass
    raise


## 4) Load the model

This notebook supports two model sources:

1. **Your own exported model** from Physical AI Studio or LeRobot. Set `MODEL_SOURCE = "local"` and point `EXPORT_DIR` to the exported OpenVINO policy package.
2. **The public example model** from Hugging Face. Set `MODEL_SOURCE = "hub"`; the notebook downloads the OpenVINO policy package and uses it for deployment.

The exported model must match your runtime setup, especially the number and names of cameras.


In [ ]:
# ============================================================
# ACTION REQUIRED - Choose model source and deployment device.
# ============================================================
# "local": use your own exported model from Physical AI Studio or LeRobot.
# "hub": download the public example OpenVINO Pi0.5 policy package.
MODEL_SOURCE = "local"  # "local" or "hub"

# Used when MODEL_SOURCE == "local".
EXPORT_DIR = "exports/pi05-pick-place-purple-cube"

# Used when MODEL_SOURCE == "hub".
MODEL_REPO_ID = "eugene123tw/pi05-pick-place-purple-cube"
DATASET_REPO_ID = "gtamir/pick-place-purple-cube"
DATASET_NAME = "pick-place-purple-cube"
DOWNLOAD_EXAMPLE_DATASET_METADATA = False
ASSETS_DIR = Path("physicalai_assets").resolve()

DEVICE = "GPU"  # OpenVINO can run on "GPU", "CPU", "NPU".
TASK = "pick up the box"


If `MODEL_SOURCE = "local"`, make sure `EXPORT_DIR` contains the full exported policy package, including `manifest.json`, model IR files, tokenizer files, and metadata.

If `MODEL_SOURCE = "hub"`, the next cell downloads the public example model package from Hugging Face. This is useful for validating the notebook before using your own model.

<img src="media/download-model.png" alt="Model download in Physical AI Studio" width="800" height="200">


In [ ]:
import openvino_tokenizers  # noqa: F401 - registers OpenVINO tokenizer custom ops
from huggingface_hub import snapshot_download
from physicalai.inference import InferenceModel

if MODEL_SOURCE == "hub":
    EXPORT_DIR = Path(
        snapshot_download(
            repo_id=MODEL_REPO_ID,
            local_dir=ASSETS_DIR / "models" / MODEL_REPO_ID.replace("/", "__"),
            allow_patterns=[
                "manifest.json",
                "pi05.xml",
                "pi05.bin",
                "tokenizer.xml",
                "tokenizer.bin",
                "metadata.yaml",
                "README.md",
            ],
            local_dir_use_symlinks=False,
        )
    ).resolve()

    if DOWNLOAD_EXAMPLE_DATASET_METADATA:
        dataset_dir = snapshot_download(
            repo_id=DATASET_REPO_ID,
            repo_type="dataset",
            local_dir=ASSETS_DIR / "datasets" / DATASET_REPO_ID.replace("/", "__"),
            allow_patterns=[f"{DATASET_NAME}/meta/**"],
            local_dir_use_symlinks=False,
        )
        print("Downloaded example dataset metadata to:", dataset_dir)
elif MODEL_SOURCE == "local":
    EXPORT_DIR = Path(EXPORT_DIR).expanduser().resolve()
else:
    raise ValueError("MODEL_SOURCE must be 'local' or 'hub'.")

required_model_files = ["manifest.json", "pi05.xml", "pi05.bin", "tokenizer.xml", "tokenizer.bin"]
missing = [name for name in required_model_files if not (EXPORT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required model package files in {EXPORT_DIR}: {missing}")

policy = InferenceModel.load(EXPORT_DIR, backend="openvino", device=DEVICE)
print(f"Model loaded from {EXPORT_DIR} on {DEVICE}")


## 5) Run the model (policy) on the robot

Use OpenVINO Physical AI to perform Inference.

In [ ]:
from physicalai.runtime import PolicyRuntime, SyncExecution

runtime = PolicyRuntime(
    robot=robot,
    model=policy,
    execution=SyncExecution(request_threshold=0.5),
    fps=FPS,
    cameras=cameras,
    task=TASK,
)

# Optional sanity check: the example Pi0.5 model expects two camera inputs.
sample_obs = runtime._build_model_input()
print("Runtime model inputs:")
for key, value in sample_obs.items():
    print(" ", key, getattr(value, "shape", None))
image_keys = [key for key in sample_obs if key.startswith("images.")]
if len(image_keys) != len(CAMERAS):
    raise RuntimeError(f"Camera input mismatch: CAMERAS has {len(CAMERAS)} entries but runtime built {len(image_keys)} image inputs.")

with runtime:
    print(f"Running policy at {FPS} fps for {DURATION_S}s - task: {TASK!r}")
    stats = runtime.run(duration_s=DURATION_S)

print(f"\nDone - {stats.steps} steps, {stats.inference_count} inferences, {stats.total_holds} holds")


## 6) Disconnect

In [ ]:
for name, cam in cameras.items():
    try:
        cam.disconnect()
        print(f"Camera '{name}' disconnected")
    except Exception as exc:
        print(f"Camera '{name}' disconnect warning: {exc}")

try:
    robot.disconnect()
    print("Robot disconnected")
except NameError:
    print("Robot was not created")
except Exception as exc:
    print(f"Robot disconnect warning: {exc}")
